# 🚀 Step 1: Building the RAG Pipeline
### Project: Evaluating the Impact of RAG on Reducing LLM Hallucinations

This notebook walks through:
1. Ingesting domain knowledge documents
2. Text chunking with sliding window overlap
3. Computing neural embeddings (`sentence-transformers/all-MiniLM-L6-v2`)
4. Storing vectors and running cosine similarity search

In [ ]:
# Install requirements if running in Google Colab
# !pip install sentence-transformers langchain chromadb pandas numpy matplotlib seaborn

import sys
from pathlib import Path

# Add parent directory to path
sys.path.append('..')

from src.data_loader import load_documents, RecursiveCharacterTextSplitter
from src.vector_store import SimpleVectorStore, EmbeddingEngine
from src.config import DOCUMENTS_DIR, VECTOR_STORE_DIR

print("Environment initialized successfully.")

## 1. Load Domain Knowledge Corpus
We load 50+ domain documents covering Space Exploration, Astronomy, and Robotic Missions.

In [ ]:
documents = load_documents(DOCUMENTS_DIR)
print(f"Total loaded documents: {len(documents)}")
print(f"Sample document source: {documents[0].metadata['source']}")
print("Sample content snippet:\n", documents[0].page_content[:300], "...")

## 2. Chunking the Documents
We split long documents into manageable chunks (~400 characters with 60-character overlap) to maintain semantic context across boundaries.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = splitter.split_documents(documents)
print(f"Total chunks generated: {len(chunks)}")
print(f"Sample chunk metadata: {chunks[0].metadata}")
print(f"Sample chunk text:\n{chunks[0].page_content}")

## 3. Embedding Generation & Vector Indexing
We embed all chunks and build an in-memory cosine similarity vector index.

In [ ]:
embedding_engine = EmbeddingEngine()
vector_store = SimpleVectorStore(embedding_engine=embedding_engine)
vector_store.add_documents(chunks)
vector_store.save(VECTOR_STORE_DIR)
print(f"Vector store indexed with {len(vector_store.documents)} chunks and saved.")

## 4. Test Semantic Retrieval
Let's test retrieving relevant context for a sample question.

In [ ]:
query = "What instrument on Perseverance demonstrated producing oxygen from Mars atmosphere?"
results = vector_store.similarity_search_with_score(query, k=3)

print(f"Query: {query}\n" + "="*60)
for idx, (doc, score) in enumerate(results, 1):
    print(f"\n[Chunk {idx}] (Similarity Score: {score:.4f}) | Source: {doc.metadata.get('source')}")
    print(doc.page_content)